# Lecture 6: Loading Models with `from_pretrained`

This notebook explores how to load pre-trained models using the `from_pretrained` method from the Hugging Face Transformers library. We also look at model configuration, weights, and the local caching mechanism.

> **Note:** Running these cells requires a GPU and a Hugging Face access token. The `meta-llama/Meta-Llama-3.1-8B-Instruct` model is gated, so make sure you have requested access on the Hugging Face Hub before running this notebook.

# Step 1: Load libraries and log in to Huggingface

In [ ]:
!pip install -q torch transformers bitsandbytes

In [ ]:
import os
import torch
from huggingface_hub import login
from dotenv import load_dotenv

load_dotenv()

# Set HF_TOKEN in your environment or a .env file
hf_token = os.getenv("HF_TOKEN")

if hf_token is None:
    raise Exception("HF_TOKEN is missing. Set it in your environment or a .env file.")

login(hf_token, add_to_git_credential=True)

# Step 2: Load quantization configuration for model

In [ ]:
from transformers import BitsAndBytesConfig

# Quantization Config - this allows us to load the model into memory and use less memory
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

# Step 3: Load a Pre-trained Model
We will use the `meta-llama/Meta-Llama-3.1-8B-Instruct` model as an example. This step demonstrates how to load the model and tokenizer.

In [ ]:
from transformers import AutoModelForCausalLM

model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", quantization_config=quant_config)

print(f"Model '{model_name}' loaded successfully!")

# Step 4: Explore the Model Configuration
The configuration of a model contains important details such as the number of layers, hidden size, and more.

In [ ]:
config = model.config
print("Model Configuration:")
print(config)

Here you can see the actual layers

In [ ]:
model

# Step 5: Understand Caching
When you load a model, it is cached locally to avoid downloading it again. The models are usually stored in the following path: ~/.cache/huggingface/hub by default

See further reference: [Huggingface cache management](~/.cache/huggingface/hub)

# Step 6: Tokenizing a Prompt and Generating Text
In this step, we will tokenize a list of messages, pass it to the model, and generate text as output.

In [ ]:
# An instruct model requires a list of messages as we saw in AI Engineering Essentials Part 1
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What is the best way to structure and organize my thoughts?"}
  ]

In [ ]:
from transformers import AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to(device)

print(inputs)

In [ ]:
# Pass the input IDs to the model to generate output
output_ids = model.generate(inputs, max_new_tokens=80)

# Display the generated output IDs
print("Generated Output IDs:", output_ids)

In [ ]:
generated_text = tokenizer.decode(output_ids[0])

print("Generated Text:", generated_text)

# Takeaways

- The `from_pretrained` method simplifies loading pre-trained models and tokenizers.
- Models are cached locally for efficiency.
- You can explore model configurations and map models to devices for optimized inference.